# Day 53 — Multi-head attention and the transformer block

One head mixes the sequence one way. A block with `h` heads mixes it `h` ways at the
same parameter count, adds a position-wise MLP, and wraps both in residual connections
and a normalisation layer. Almost every way of getting that slightly wrong produces an
array of exactly the right shape.

In [1]:
import math

import numpy as np

# ================================================================ 1. attention
def softmax(x, axis=-1):
    x = np.asarray(x, dtype=float)
    m = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - m)
    return e / e.sum(axis=axis, keepdims=True)


def attention(Q, K, V, mask=None):
    """Day 52's four lines, now applied per head."""
    d = Q.shape[-1]
    s = Q @ np.swapaxes(K, -1, -2) / math.sqrt(d)
    if mask is not None:
        s = np.where(mask, s, -np.inf)
    w = softmax(s, axis=-1)
    return w @ V, w


# ================================================================ 2. splitting heads
def split_heads(X, h):
    """(n, d_model) -> (h, n, d_head), the correct way.

    Head j owns the contiguous slice X[:, j*d_head:(j+1)*d_head].  Reshaping to
    (n, h, d_head) groups the feature axis first and then swaps it to the front, so each
    head keeps its own block of features for every position.
    """
    n, d = X.shape
    return X.reshape(n, h, d // h).transpose(1, 0, 2)


def split_heads_wrong(X, h):
    """(n, d_model) -> (h, n, d_head) by reshaping the wrong axis first.

    This hands head 0 the first n/h POSITIONS instead of the first d/h FEATURES.  The
    shape is identical, every later line runs, and the gradients flow.
    """
    n, d = X.shape
    return X.reshape(h, n, d // h)


def merge_heads(H):
    """(h, n, d_head) -> (n, d_model), the inverse of split_heads."""
    h, n, dh = H.shape
    return H.transpose(1, 0, 2).reshape(n, h * dh)


def multi_head_attention(X, Wq, Wk, Wv, Wo, h, mask=None, split=split_heads):
    """The layer: project, split into h heads, attend per head, concatenate, project."""
    Q, K, V = split(X @ Wq, h), split(X @ Wk, h), split(X @ Wv, h)
    out, w = attention(Q, K, V, mask=mask)
    return merge_heads(out) @ Wo, w


X = np.round(np.random.default_rng(3).standard_normal((4, 8)), 2)
print('input', X.shape, '-> 2 heads')
print('split_heads(X, 2)[0]  = features 0..3 of every token')
print(split_heads(X, 2)[0])
print()
print('X.reshape(2, 4, 4)[0] = tokens 0..1, all their features')
print(X.reshape(2, 4, 4)[0])
print()
print('round trip:', np.allclose(merge_heads(split_heads(X, 2)), X))

input (4, 8) -> 2 heads
split_heads(X, 2)[0]  = features 0..3 of every token
[[ 2.04 -2.56  0.42 -0.57]
 [-0.87  3.32  0.23 -0.35]
 [ 0.48 -0.24  0.96 -0.2 ]
 [-0.18  0.54  1.94 -0.27]]

X.reshape(2, 4, 4)[0] = tokens 0..1, all their features
[[ 2.04 -2.56  0.42 -0.57]
 [-0.45 -0.22 -2.02 -0.23]
 [-0.87  3.32  0.23 -0.35]
 [-0.28 -0.67 -1.06 -0.39]]

round trip: True


## h heads cost nothing extra

The head count never changes the four `d_model x d_model` projections; it changes how
they are cut. Each head then produces its own `n x n` weight matrix, so the layer mixes
the sequence `h` different ways for the same parameters.

In [2]:
def make_params(d_model=64, h=8, seed=0, scale=None):
    """He-ish initialisation: every projection scaled by 1/sqrt(fan_in)."""
    rng = np.random.default_rng(seed)
    s = (1 / math.sqrt(d_model)) if scale is None else scale
    d_ff = 4 * d_model
    return {'h': h,
            'Wq': rng.standard_normal((d_model, d_model)) * s,
            'Wk': rng.standard_normal((d_model, d_model)) * s,
            'Wv': rng.standard_normal((d_model, d_model)) * s,
            'Wo': rng.standard_normal((d_model, d_model)) * s,
            'W1': rng.standard_normal((d_model, d_ff)) * s,
            'bb1': np.zeros(d_ff),
            'W2': rng.standard_normal((d_ff, d_model)) / math.sqrt(d_ff),
            'bb2': np.zeros(d_model),
            'g1': np.ones(d_model), 'b1': np.zeros(d_model),
            'g2': np.ones(d_model), 'b2': np.zeros(d_model)}


def head_params(d_model, h):
    """Four d_model x d_model matrices, whatever h is - h only changes how they split."""
    return {'per_head_dim': d_model // h,
            'Wq': d_model * d_model, 'total': 4 * d_model * d_model,
            'score_entries_per_layer': h}


def split_output_gap(n=16, d_model=64, h=8, seed=5):
    """How far apart the two splittings put the layer's output."""
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n, d_model))
    p = make_params(d_model, h, seed=seed)
    good, _ = multi_head_attention(X, p['Wq'], p['Wk'], p['Wv'], p['Wo'], h)
    bad, _ = multi_head_attention(X, p['Wq'], p['Wk'], p['Wv'], p['Wo'], h,
                                  split=split_heads_wrong)
    return (float(np.abs(good - bad).max()),
            float(np.linalg.norm(good - bad) / np.linalg.norm(good)),
            good.shape == bad.shape)


for h in (1, 2, 4, 8, 16, 64):
    r = head_params(64, h)
    print(f"h={h:>3}  d_head={r['per_head_dim']:>3}  params={r['total']:,}  "
          f"weight matrices per layer={r['score_entries_per_layer']}")
print()
mx, rel, same_shape = split_output_gap()
print('wrong split, same shape:', same_shape)
print(f'layer output differs by max {mx:.3f}, {rel * 100:.0f}% of the output norm')

h=  1  d_head= 64  params=16,384  weight matrices per layer=1
h=  2  d_head= 32  params=16,384  weight matrices per layer=2
h=  4  d_head= 16  params=16,384  weight matrices per layer=4
h=  8  d_head=  8  params=16,384  weight matrices per layer=8
h= 16  d_head=  4  params=16,384  weight matrices per layer=16
h= 64  d_head=  1  params=16,384  weight matrices per layer=64

wrong split, same shape: True
layer output differs by max 1.911, 153% of the output norm


## What the extra heads buy

A target where one half of the channels wants the previous token and the other half
wants the first token, measured over 12 different inputs. Two heads do it exactly. The
single head is fitted by gradient descent and given everything a real head has and
more: its own `d x d` channel matrix (what `Wv` and `Wo` amount to) plus a free choice
of attention matrix per input. Without that channel matrix the comparison is rigged —
at `n = d = 8` one input can be fitted exactly by solving for the channel matrix alone.

In [3]:
# ================================================================ 3. what h heads buy
def two_pattern_target(n=8, d=8, seed=0):
    """A target this block has to produce: one half of the channels wants the previous
    token, the other half wants the first token.

    Both are things attention can do.  The question is whether ONE head can do both at
    once, because a head applies the same weight matrix to every channel it carries.
    """
    rng = np.random.default_rng(seed)
    V = rng.standard_normal((n, d))
    prev = np.eye(n, k=-1)
    prev[0, 0] = 1.0                       # position 0 has no previous token
    first = np.zeros((n, n))
    first[:, 0] = 1.0
    target = np.empty_like(V)
    target[:, :d // 2] = (prev @ V)[:, :d // 2]
    target[:, d // 2:] = (first @ V)[:, d // 2:]
    return V, prev, first, target


def best_single_head(m=12, n=8, d=8, steps=4000, lr=0.5, seed=0):
    """The best a single head could do, given everything a real head has.

    A real head computes A (X Wv) Wo = A X M: one attention matrix A, and one channel
    matrix M that can route features freely.  This fits both - M shared across m
    different inputs, and A chosen per input, which is more than a real head gets
    (its A has to come out of Q and K).  The one thing it cannot do is use a different
    A for different channels, and that is the whole point.
    """
    rng = np.random.default_rng(seed)
    Vs, Ts, Ls = [], [], []
    prev = np.eye(n, k=-1)
    prev[0, 0] = 1.0
    first = np.zeros((n, n))
    first[:, 0] = 1.0
    for t in range(m):
        V = rng.standard_normal((n, d))
        T = np.empty_like(V)
        T[:, :d // 2] = (prev @ V)[:, :d // 2]
        T[:, d // 2:] = (first @ V)[:, d // 2:]
        Vs.append(V)
        Ts.append(T)
        Ls.append(rng.standard_normal((n, n)) * 0.1)
    M = np.eye(d)
    size = m * n * d
    for _ in range(steps):
        dM = np.zeros_like(M)
        grads = []
        for V, T, L in zip(Vs, Ts, Ls):
            A = softmax(L, axis=-1)
            Z = A @ V
            R = Z @ M - T
            dM += 2 * Z.T @ R / size
            dZ = 2 * R @ M.T / size
            dA = dZ @ V.T
            grads.append(A * (dA - (dA * A).sum(axis=-1, keepdims=True)))
        for L, g in zip(Ls, grads):
            L -= lr * g
        M -= lr * dM
    err = 0.0
    for V, T, L in zip(Vs, Ts, Ls):
        err += float(np.mean((softmax(L, axis=-1) @ V @ M - T) ** 2))
    return softmax(Ls[0], axis=-1), M, err / m


def one_input_exact(n=8, d=8, seed=0):
    """Why the comparison above needs more than one input.

    With a single input and n == d, the matrix A @ V is square and invertible, so
    solving (A V) M = target for M alone fits the target exactly for essentially any
    attention matrix A.  A result like that is about the toy, not about heads.
    """
    V, prev, first, target = two_pattern_target(n, d, seed)
    A = softmax(np.random.default_rng(seed + 1).standard_normal((n, n)), axis=-1)
    M = np.linalg.solve(A @ V, target)
    return float(np.mean((A @ V @ M - target) ** 2))


def two_head_error(V, prev, first, target):
    """Two heads, each carrying half the channels, reproduce the target exactly."""
    out = np.empty_like(V)
    half = V.shape[1] // 2
    out[:, :half] = (prev @ V)[:, :half]
    out[:, half:] = (first @ V)[:, half:]
    return float(np.mean((out - target) ** 2))


def two_head_error_many(m=12, n=8, d=8, seed=0):
    """The same m inputs through two heads with fixed patterns, measured the same way."""
    rng = np.random.default_rng(seed)
    prev = np.eye(n, k=-1)
    prev[0, 0] = 1.0
    first = np.zeros((n, n))
    first[:, 0] = 1.0
    err = 0.0
    for _ in range(m):
        V = rng.standard_normal((n, d))
        T = np.empty_like(V)
        T[:, :d // 2] = (prev @ V)[:, :d // 2]
        T[:, d // 2:] = (first @ V)[:, d // 2:]
        out = np.empty_like(V)
        out[:, :d // 2] = (prev @ V)[:, :d // 2]
        out[:, d // 2:] = (first @ V)[:, d // 2:]
        err += float(np.mean((out - T) ** 2))
    return err / m


def target_scale(m=12, n=8, d=8, seed=0):
    """What predicting zero everywhere would score on the same m inputs."""
    rng = np.random.default_rng(seed)
    prev = np.eye(n, k=-1)
    prev[0, 0] = 1.0
    first = np.zeros((n, n))
    first[:, 0] = 1.0
    err = 0.0
    for _ in range(m):
        V = rng.standard_normal((n, d))
        T = np.empty_like(V)
        T[:, :d // 2] = (prev @ V)[:, :d // 2]
        T[:, d // 2:] = (first @ V)[:, d // 2:]
        err += float(np.mean(T ** 2))
    return err / m


A, M, single = best_single_head()
print(f'two heads, one pattern each:          MSE {two_head_error_many():.2e}')
print(f'one head, with a channel matrix too:  MSE {single:.4f}')
print(f'predicting zero everywhere:           MSE {target_scale():.4f}')
print()
print(f'one input alone, solving for the channel matrix: MSE {one_input_exact():.1e}')
print('-> which is why the comparison above uses 12 inputs')

two heads, one pattern each:          MSE 0.00e+00
one head, with a channel matrix too:  MSE 0.1446
predicting zero everywhere:           MSE 0.8746

one input alone, solving for the channel matrix: MSE 5.7e-30
-> which is why the comparison above uses 12 inputs


## LayerNorm, and the one-pass variance that keeps being written

`var = E[x^2] - E[x]^2` takes one pass instead of two. A residual stream accumulates a
mean offset, and that identity then subtracts two large numbers to get a small one.

In [4]:
# ================================================================ 4. normalisation
def layer_norm(x, gamma=None, beta=None, eps=1e-5):
    """Centre, scale to unit variance, then rescale with learned gamma and beta."""
    mu = x.mean(axis=-1, keepdims=True)
    var = ((x - mu) ** 2).mean(axis=-1, keepdims=True)
    y = (x - mu) / np.sqrt(var + eps)
    if gamma is not None:
        y = y * gamma
    if beta is not None:
        y = y + beta
    return y


def layer_norm_naive_var(x, eps=1e-5):
    """The same layer, with the variance computed as E[x^2] - E[x]^2.

    One pass instead of two, which is why it keeps being written - and two large numbers
    subtracted to get a small one, which is why it stops working.
    """
    mu = x.mean(axis=-1, keepdims=True)
    var = (x ** 2).mean(axis=-1, keepdims=True) - mu ** 2
    return (x - mu) / np.sqrt(var + eps), var


def rms_norm(x, gamma=None, eps=1e-5):
    """RMSNorm: divide by the root mean square, never subtract the mean."""
    r = np.sqrt((x ** 2).mean(axis=-1, keepdims=True) + eps)
    y = x / r
    return y * gamma if gamma is not None else y


def cancellation_table(offsets=(0.0, 1e2, 1e3, 1e4, 1e5, 1e6), n=512, seed=0, dtype=np.float32):
    """How a constant offset - exactly what a residual stream accumulates - destroys the
    one-pass variance while leaving the two-pass one alone."""
    rng = np.random.default_rng(seed)
    base = rng.standard_normal(n).astype(dtype)
    rows = []
    for off in offsets:
        x = (base + dtype(off)).astype(dtype)
        two = float(((x - x.mean()) ** 2).mean())
        one = float((x.astype(dtype) ** 2).mean() - x.mean() ** 2)
        rows.append({'offset': off, 'two_pass': two, 'one_pass': one,
                     'distinct': int(len(np.unique(x))),
                     'rel_err': abs(one - two) / two})
    return rows


print(f"{'offset':>10} {'two-pass':>12} {'one-pass':>14} {'distinct float32':>18}")
for r in cancellation_table():
    print(f"{r['offset']:>10,.0f} {r['two_pass']:>12.6f} {r['one_pass']:>14.6f} "
          f"{r['distinct']:>18}")
print()
Z = np.random.default_rng(4).standard_normal((3, 64)) * 2 + 7
print('layer_norm: mean %.2e, std %.4f' % (layer_norm(Z)[0].mean(), layer_norm(Z)[0].std()))
print('rms_norm:   mean %+.4f, std %.4f' % (rms_norm(Z)[0].mean(), rms_norm(Z)[0].std()))
print('shift by +100 changes layer_norm by %.2e and rms_norm by %.4f'
      % (np.abs(layer_norm(Z) - layer_norm(Z + 100)).max(),
         np.abs(rms_norm(Z) - rms_norm(Z + 100)).max()))

    offset     two-pass       one-pass   distinct float32
         0     1.019437       1.019437                512
       100     1.019437       1.018555                512
     1,000     1.019434       1.062500                511
    10,000     1.019408      -8.000000                475
   100,000     1.019417    1024.000000                304
 1,000,000     1.020317       0.000000                 77

layer_norm: mean 2.64e-16, std 1.0000
rms_norm:   mean +0.9628, std 0.2701
shift by +100 changes layer_norm by 6.13e-15 and rms_norm by 0.8014


## The block, pre-LN and post-LN

Post-LN normalises after the residual add, so the path from input to output goes
through a norm in every block, and the norm of the stream stops depending on the depth
(with `gamma = 1` and `beta = 0` as initialised here, it lands on `sqrt(n * d_model)`). Pre-LN normalises only what each branch reads, leaving
one unnormalised path all the way up — which is why a pre-LN stack ends with one final
LayerNorm before the output head.

In [5]:
# ================================================================ 5. the block
def gelu(x):
    """The tanh approximation, which is what the original implementations shipped."""
    return 0.5 * x * (1 + np.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * x ** 3)))


def relu(x):
    return np.maximum(x, 0.0)


def ffn(x, W1, b1, W2, b2, act=gelu):
    """Position-wise MLP: expand by 4x, non-linearity, project back."""
    return act(x @ W1 + b1) @ W2 + b2


def block_pre_ln(x, p, mask=None):
    """Pre-LN: normalise the branch input, leave the residual path untouched."""
    a, _ = multi_head_attention(layer_norm(x, p['g1'], p['b1']), p['Wq'], p['Wk'],
                                p['Wv'], p['Wo'], p['h'], mask=mask)
    x = x + a
    return x + ffn(layer_norm(x, p['g2'], p['b2']), p['W1'], p['bb1'], p['W2'], p['bb2'])


def block_post_ln(x, p, mask=None):
    """Post-LN: normalise after the add, so the residual path goes through the norm."""
    a, _ = multi_head_attention(x, p['Wq'], p['Wk'], p['Wv'], p['Wo'], p['h'], mask=mask)
    x = layer_norm(x + a, p['g1'], p['b1'])
    return layer_norm(x + ffn(x, p['W1'], p['bb1'], p['W2'], p['bb2']), p['g2'], p['b2'])


def depth_signal(layers=32, n=16, d_model=64, h=8, seed=0, eps=1e-3):
    """Push a tiny perturbation through L blocks and watch how much of it survives.

    How far a change at the input still moves the output is the forward-mode mirror of
    how much gradient gets back to the first layer, and it needs no autograd.
    """
    rng = np.random.default_rng(seed)
    x0 = rng.standard_normal((n, d_model))
    dx = rng.standard_normal((n, d_model))
    dx = eps * dx / np.linalg.norm(dx)
    ps = [make_params(d_model, h, seed=100 + i) for i in range(layers)]
    out = {}
    for name, fn in (('pre', block_pre_ln), ('post', block_post_ln)):
        a, b = x0.copy(), x0 + dx
        trace = []
        for p in ps:
            a, b = fn(a, p), fn(b, p)
            amp = float(np.linalg.norm(b - a) / np.linalg.norm(dx))
            trace.append({'norm': float(np.linalg.norm(a)), 'signal': amp,
                          'relative': amp / float(np.linalg.norm(a))})
        out[name] = trace
    return out


def residual_growth(layers=32, n=16, d_model=64, h=8, seed=0, branch_scale=None):
    """The norm of the residual stream before the first block and after each one,
    with and without the 1/sqrt(2L) scaling applied to what each branch writes."""
    rng = np.random.default_rng(seed)
    x = rng.standard_normal((n, d_model))
    x = x / np.linalg.norm(x) * math.sqrt(n * d_model)
    ps = [make_params(d_model, h, seed=200 + i) for i in range(layers)]
    s = 1.0 if branch_scale is None else branch_scale
    start = float(np.linalg.norm(x))
    norms = []
    for p in ps:
        a, _ = multi_head_attention(layer_norm(x), p['Wq'], p['Wk'], p['Wv'], p['Wo'], p['h'])
        x = x + s * a
        x = x + s * ffn(layer_norm(x), p['W1'], p['bb1'], p['W2'], p['bb2'])
        norms.append(float(np.linalg.norm(x)))
    return {'start': start, 'norms': norms}


tr = depth_signal(layers=32)
print(f"{'after block':>12} {'||x|| pre':>10} {'||x|| post':>11} {'influence pre':>14} {'post':>7}")
for i in (0, 7, 15, 31):
    a, b = tr['pre'][i], tr['post'][i]
    print(f"{i + 1:>12} {a['norm']:>10.1f} {b['norm']:>11.1f} {a['signal']:>14.3f} "
          f"{b['signal']:>7.3f}")
print()
plain = residual_growth(layers=32)
scaled = residual_growth(layers=32, branch_scale=1 / math.sqrt(64))
print(f"the stream starts at {plain['start']:.1f}")
print(f"after 32 blocks: {plain['norms'][-1]:.1f} unscaled, {scaled['norms'][-1]:.1f} "
      f"with 1/sqrt(2L) on each branch")
print(f"growth {plain['norms'][-1] / plain['start']:.1f}x against "
      f"sqrt(32) = {math.sqrt(32):.2f}")

 after block  ||x|| pre  ||x|| post  influence pre    post
           1       37.9        32.0          1.355   1.134
           8       81.9        32.0          3.580   1.441
          16      127.1        32.0          5.599   1.541
          32      197.0        32.0          7.505   1.104

the stream starts at 32.0
after 32 blocks: 181.8 unscaled, 35.8 with 1/sqrt(2L) on each branch
growth 5.7x against sqrt(32) = 5.66


## Where the parameters and the FLOPs are

Two thirds of a block is the MLP. The quadratic part of the attention overtakes
everything else at a context of `6 * d_model`, and the head count changes neither.

In [6]:
# ================================================================ 6. the budget
def block_params(d_model, d_ff=None, h=8):
    """Parameters in one block, split between attention and the MLP."""
    d_ff = 4 * d_model if d_ff is None else d_ff
    attn = 4 * d_model * d_model
    mlp = 2 * d_model * d_ff
    return {'attn': attn, 'mlp': mlp, 'total': attn + mlp,
            'mlp_share': mlp / (attn + mlp), 'd_ff': d_ff, 'h': h}


def block_flops(n, d_model, d_ff=None):
    """Multiply-accumulates for one block on a sequence of n tokens.

    The projections and the MLP are linear in n.  The two attention matmuls (scores and
    the weighted sum) are quadratic, and they do not depend on the number of heads.
    """
    d_ff = 4 * d_model if d_ff is None else d_ff
    linear = n * (4 * d_model * d_model + 2 * d_model * d_ff)
    quad = 2 * n * n * d_model
    return {'n': n, 'linear': linear, 'quad': quad, 'total': linear + quad,
            'quad_share': quad / (linear + quad)}


def flop_crossover(d_model, d_ff=None):
    """The context length at which the quadratic part catches the linear part."""
    d_ff = 4 * d_model if d_ff is None else d_ff
    return (4 * d_model * d_model + 2 * d_model * d_ff) / (2 * d_model)


def budget_table(configs=((768, 12), (2048, 16), (4096, 32), (8192, 64))):
    rows = []
    for d, h in configs:
        p = block_params(d, h=h)
        rows.append({'d_model': d, 'h': h, 'attn': p['attn'], 'mlp': p['mlp'],
                     'total': p['total'], 'mlp_share': p['mlp_share'],
                     'crossover': flop_crossover(d)})
    return rows


def flop_table(d_model=4096, ns=(512, 2048, 8192, 32768)):
    return [block_flops(n, d_model) for n in ns]


for r in budget_table():
    print(f"d_model {r['d_model']:>5}  attention {r['attn']:>12,}  MLP {r['mlp']:>12,}  "
          f"MLP share {r['mlp_share'] * 100:.0f}%  crossover {r['crossover']:,.0f}")
print()
for r in flop_table():
    print(f"context {r['n']:>7,}  linear {r['linear']:>18,}  quadratic {r['quad']:>18,}  "
          f"quadratic share {r['quad_share'] * 100:5.1f}%")

d_model   768  attention    2,359,296  MLP    4,718,592  MLP share 67%  crossover 4,608
d_model  2048  attention   16,777,216  MLP   33,554,432  MLP share 67%  crossover 12,288
d_model  4096  attention   67,108,864  MLP  134,217,728  MLP share 67%  crossover 24,576
d_model  8192  attention  268,435,456  MLP  536,870,912  MLP share 67%  crossover 49,152

context     512  linear    103,079,215,104  quadratic      2,147,483,648  quadratic share   2.0%
context   2,048  linear    412,316,860,416  quadratic     34,359,738,368  quadratic share   7.7%
context   8,192  linear  1,649,267,441,664  quadratic    549,755,813,888  quadratic share  25.0%
context  32,768  linear  6,597,069,766,656  quadratic  8,796,093,022,208  quadratic share  57.1%


## LeetCode 189 — Rotate Array

Reverse the whole array, then reverse the two pieces. The cycle method is the other
in-place answer, and it needs `gcd(n, k)` starting points: start only at index 0 and it
moves one cycle and leaves the rest alone.

In [7]:
# ================================================================ 7. LeetCode 189
class SolutionRotateArray:
    """LeetCode 189 - Rotate Array.

    Reverse the whole array, then reverse each of the two pieces.  No extra array, and
    exactly the kind of in-place index surgery that splitting heads is: the bytes do not
    move for a reshape, they do move for a transpose, and knowing which is which is the
    difference between a free operation and a copy.
    """

    def rotate(self, nums: list[int], k: int) -> None:
        n = len(nums)
        k %= n                                  # k can exceed n
        self._rev(nums, 0, n - 1)
        self._rev(nums, 0, k - 1)
        self._rev(nums, k, n - 1)

    @staticmethod
    def _rev(a, i, j):
        while i < j:
            a[i], a[j] = a[j], a[i]
            i, j = i + 1, j - 1


def rotate_cycles(nums, k):
    """The other in-place method: walk the permutation cycles.

    Correct, and it needs gcd(n, k) separate starting points - forget that and you move
    one cycle and leave the rest untouched.
    """
    n = len(nums)
    k %= n
    if k == 0:
        return nums
    moved = 0
    start = 0
    while moved < n:
        cur, val = start, nums[start]
        while True:
            nxt = (cur + k) % n
            nums[nxt], val = val, nums[nxt]
            cur = nxt
            moved += 1
            if cur == start:
                break
        start += 1
    return nums


def rotate_one_cycle(nums, k):
    """The same walk started only at index 0: right for some k, silently partial for
    others, because it visits only one of the gcd(n, k) cycles."""
    n = len(nums)
    k %= n
    if k == 0:
        return nums
    cur, val = 0, nums[0]
    while True:
        nxt = (cur + k) % n
        nums[nxt], val = val, nums[nxt]
        cur = nxt
        if cur == 0:
            break
    return nums


b = [1, 2, 3, 4, 5, 6, 7]
SolutionRotateArray().rotate(b, 3)
print('[1..7], k=3 ->', b)
c = [1, 2, 3, 4, 5, 6]
print('all cycles,  k=2 ->', rotate_cycles(list(c), 2))
print('one cycle,   k=2 ->', rotate_one_cycle(list(c), 2), '  (gcd(6, 2) = 2 cycles)')
print('one cycle,   k=1 ->', rotate_one_cycle(list(c), 1), '  (gcd(6, 1) = 1 cycle)')

[1..7], k=3 -> [5, 6, 7, 1, 2, 3, 4]
all cycles,  k=2 -> [5, 6, 1, 2, 3, 4]
one cycle,   k=2 -> [5, 2, 1, 4, 3, 6]   (gcd(6, 2) = 2 cycles)
one cycle,   k=1 -> [6, 1, 2, 3, 4, 5]   (gcd(6, 1) = 1 cycle)


## LeetCode 303 — Range Sum Query (immutable)

One prefix-sum pass turns every later range query into one subtraction. Extend the same
trick to variance and it meets the cancellation from the LayerNorm section.

In [8]:
# ================================================================ 8. LeetCode 303
class NumArray:
    """LeetCode 303 - Range Sum Query (immutable).

    One prefix-sum pass up front turns every later range query into one subtraction.
    """

    def __init__(self, nums: list[int]):
        self.pre = [0]
        for v in nums:
            self.pre.append(self.pre[-1] + v)

    def sumRange(self, left: int, right: int) -> int:
        return self.pre[right + 1] - self.pre[left]


def window_variance_prefix(x, lo, hi):
    """Variance of a window from prefix sums of x and x^2 - one subtraction each.

    It is the same identity LayerNorm's one-pass variance uses, and it fails the same
    way: two large numbers differenced to produce a small one.
    """
    x = np.asarray(x, dtype=np.float32)
    p1 = np.concatenate([[0], np.cumsum(x, dtype=np.float32)])
    p2 = np.concatenate([[0], np.cumsum(x * x, dtype=np.float32)])
    m = hi - lo
    s1 = p1[hi] - p1[lo]
    s2 = p2[hi] - p2[lo]
    return float(s2 / m - (s1 / m) ** 2)


nums = [-2, 0, 3, -5, 2, -1]
na = NumArray(nums)
print('sumRange(0,2) =', na.sumRange(0, 2), ' sumRange(2,5) =', na.sumRange(2, 5),
      ' sumRange(0,5) =', na.sumRange(0, 5))
print()
x = np.full(512, 1e4, dtype=np.float32) + np.random.default_rng(0).standard_normal(512).astype(np.float32)
print('window variance, computed directly: %.6f' % float(np.var(x[100:300].astype(np.float64))))
print('from float32 prefix sums:           %.6f' % window_variance_prefix(x, 100, 300))

sumRange(0,2) = 1  sumRange(2,5) = -1  sumRange(0,5) = -3

window variance, computed directly: 1.080859
from float32 prefix sums:           28.860555


## The assertions

Everything above is checked here: splitting and merging are inverses, one head of the
full width is plain single-head attention, LayerNorm centres and RMSNorm does not, the
one-pass variance goes negative, the FLOP crossover is where the two halves are equal,
and both LeetCode solutions agree with brute force on 300 random cases each.

In [9]:
# ================================================================ 10. assertions
def check():
    rng = np.random.default_rng(0)

    # splitting and merging are inverses, and the correct split keeps feature blocks
    X = rng.standard_normal((6, 12))
    assert np.allclose(merge_heads(split_heads(X, 4)), X)
    assert np.allclose(split_heads(X, 4)[2], X[:, 6:9])          # head 2 owns features 6..8
    assert split_heads_wrong(X, 4).shape == split_heads(X, 4).shape
    assert not np.allclose(split_heads_wrong(X, 4), split_heads(X, 4))

    # one head of the full width is plain single-head attention
    d = 16
    p = make_params(d, 1, seed=1)
    Y = rng.standard_normal((5, d))
    one, w = multi_head_attention(Y, p['Wq'], p['Wk'], p['Wv'], p['Wo'], 1)
    ref, wref = attention(Y @ p['Wq'], Y @ p['Wk'], Y @ p['Wv'])
    assert np.allclose(one, ref @ p['Wo']) and np.allclose(w[0], wref)

    # layer norm really does centre and scale; RMSNorm does not centre
    Z = rng.standard_normal((7, 32)) * 3 + 5
    ln = layer_norm(Z)
    assert np.allclose(ln.mean(axis=-1), 0, atol=1e-8)
    assert np.allclose(ln.std(axis=-1), 1, atol=1e-4)
    assert abs(rms_norm(Z).mean()) > 1e-3
    # both are scale invariant, only layer norm is shift invariant
    assert np.allclose(layer_norm(Z), layer_norm(2 * Z), atol=1e-6)
    assert np.allclose(layer_norm(Z), layer_norm(Z + 100), atol=1e-4)
    assert np.allclose(rms_norm(Z), rms_norm(2 * Z), atol=1e-6)
    assert not np.allclose(rms_norm(Z), rms_norm(Z + 100), atol=1e-2)

    # the one-pass variance goes negative once the offset is large enough
    rows = cancellation_table()
    assert rows[0]['rel_err'] < 1e-9                 # identical with no offset
    assert any(r['one_pass'] < 0 for r in rows)      # and negative once it is large
    assert all(abs(r['two_pass'] - rows[0]['two_pass']) < 1e-2 for r in rows)

    # two heads reach the two-pattern target exactly, one head cannot - and the single
    # head is given a channel matrix as well, which is the only extra a real head has
    V, prev, first, target = two_pattern_target()
    assert two_head_error(V, prev, first, target) < 1e-20
    assert two_head_error_many() < 1e-20
    _, _, single = best_single_head(steps=400)
    assert single > 1e-2 and single < target_scale()
    assert one_input_exact() < 1e-20                   # one input proves nothing

    # a block preserves shape, and the FLOP crossover is where the two halves are equal
    pb = make_params(32, 4, seed=2)
    xb = rng.standard_normal((9, 32))
    assert block_pre_ln(xb, pb).shape == xb.shape
    assert block_post_ln(xb, pb).shape == xb.shape
    nc = flop_crossover(4096)
    f = block_flops(int(round(nc)), 4096)
    assert abs(f['linear'] - f['quad']) / f['linear'] < 1e-6

    # LeetCode 189 against the obvious slice, including k > n
    for seed in range(300):
        r = np.random.default_rng(seed)
        nums = r.integers(-50, 50, int(r.integers(1, 12))).tolist()
        k = int(r.integers(0, 25))
        want = nums[-(k % len(nums)):] + nums[:-(k % len(nums))] if k % len(nums) else list(nums)
        got = list(nums)
        SolutionRotateArray().rotate(got, k)
        assert got == want
        assert rotate_cycles(list(nums), k) == want
    a = [1, 2, 3, 4, 5, 6]
    assert rotate_one_cycle(list(a), 2) != [5, 6, 1, 2, 3, 4]    # gcd(6, 2) = 2 cycles
    assert rotate_one_cycle(list(a), 1) == [6, 1, 2, 3, 4, 5]    # gcd(6, 1) = 1 cycle

    # LeetCode 303 against the obvious sum
    for seed in range(300):
        r = np.random.default_rng(seed + 7)
        nums = r.integers(-100, 100, int(r.integers(1, 20))).tolist()
        na = NumArray(nums)
        for _ in range(5):
            i = int(r.integers(0, len(nums)))
            j = int(r.integers(i, len(nums)))
            assert na.sumRange(i, j) == sum(nums[i:j + 1])
    return True


print('all assertions passed' if check() else 'FAILED')

all assertions passed
